# Imports and data

In [1]:
import pandas as pd

from demand.comparison import baseline_predictions, make_folds, net_predictions, tabular_predictions
from demand.config import PROCESSED
from demand.data import clean, load_subset
from demand.features import build_features
from demand.models import ScaledTarget, make_lgbm, make_ridge, make_xgb
from demand.sequences import make_windows
from demand.validation import split_holdout

features = build_features(clean(load_subset()))
history_start = features.date.min()
train_part, holdout = split_holdout(features)
folds = make_folds(train_part, holdout)

# Predictions

## Baseline

In [2]:
baseline = baseline_predictions(features, folds, history_start)
print(len(folds), baseline.groupby("fold").size().to_dict())

9 {1: 650, 2: 700, 3: 700, 4: 700, 5: 700, 6: 700, 7: 700, 8: 700, 9: 700}


## Ridge, XGBoost and LightGBM

In [3]:
ridge = tabular_predictions(make_ridge, features, folds, history_start, "ridge")
xgb = tabular_predictions(lambda: ScaledTarget(make_xgb), features, folds, history_start, "xgb_scaled")
lgbm = tabular_predictions(lambda: ScaledTarget(make_lgbm), features, folds, history_start, "lgbm_scaled")

## RNN and LSTM

In [4]:
windows = make_windows(features)
rnn = net_predictions("rnn", windows, features, folds, "rnn")
lstm = net_predictions("lstm", windows, features, folds, "lstm")

# Save predictions

In [5]:
predictions = pd.concat([baseline, ridge, xgb, lgbm, rnn, lstm], ignore_index=True)
predictions.to_parquet(PROCESSED / "predictions.parquet", index=False)
predictions.groupby(["model", "fold"]).size().unstack()

fold,1,2,3,4,5,6,7,8,9
model,,,,,,,,,
lgbm_scaled,650,700,700,700,700,700,700,700,700
lstm,650,700,700,700,700,700,700,700,700
ridge,650,700,700,700,700,700,700,700,700
rnn,650,700,700,700,700,700,700,700,700
seasonal_median_4w,650,700,700,700,700,700,700,700,700
xgb_scaled,650,700,700,700,700,700,700,700,700
